In [136]:
from pathlib import Path
import pandas as pd
import numpy as np

DATAPATH = Path("/Users/sebastianodutola/Projects/Carbon_Intensity/Data/backtest_data.py")
df = pd.read_parquet(DATAPATH)
# drop duplicates that might have wiggled through the net
df = df.drop_duplicates(subset="from")
df["from"] = df["from"].apply(np.datetime64)
df["to"] = df["to"].apply(np.datetime64)
df = df.reset_index(drop=True)
df.sort_values("from")

# forward fill for all uniform timesteps
df = df.set_index("from")
full_index = pd.date_range(start=df.index.min(), end=df.index.max(), freq="30min")
df = df.reindex(full_index, method="ffill")
df = df.rename_axis("from").reset_index()

# normalise json rows
cols = pd.json_normalize(df["intensity"])
df = pd.concat([df.drop(columns="intensity"), cols], axis=1)

# forward fill NaN columns in "forecast" and "actual"
df = df.ffill()

/Users/sebastianodutola/.virtualenvs/datascience/lib/python3.11/site-packages/pandas/core/algorithms.py:1743: UserWarning: no explicit representation of timezones available for np.datetime64
  return lib.map_infer(values, mapper, convert=convert)


In [158]:
from scipy.optimize import linprog
# Optimiser for LP requires 
# c, A, b, l, u for min c^tx, Ax <= b, l < x < u 

# S = max charging speed 
# C = max_capacity
# delta_k = t_k+1 - t_k
# Delta_i,j = (delta_i for i <= j 0 otherwise)
# d_k = discharge_power in [t_k, t_k+1)
# c_k = carbon_cost delta_k
# A = (Delta)
#     (-Delta)
# b = 1/s (C1 + Delta d - x_01)
#         (x_01 - Delta d     )
# 0 < x_i < 1 if available x_i = 0 otherwise 
# Note: this optimisation algorithm doesn't optimise for minimum charge time (possible but with an extra optimisation iteration).

def optimal_discharge(x0: float, S: float, C: float, delta: np.ndarray, d: np.ndarray, carbon_intensity: np.ndarray, a: np.ndarray):
    assert delta.size == d.size == carbon_intensity.size
    # Catch edge case where empty arrays are passed to the optimisation algorithm
    if delta.size == 0:
        return (0, 0)
    n = delta.size
    Delta = np.tri(n, dtype=delta.dtype) * delta

    A = np.zeros((2 * n, n), dtype=delta.dtype)
    A[:n, :] = Delta
    A[n:, :] = -Delta

    b = np.zeros((2 * n), dtype=delta.dtype)
    b[:n] = (C - x0) * np.ones(n) + Delta @ d
    b[n:] = x0 * np.ones(n) - Delta @ d
    b /= S

    c = delta * carbon_intensity 
    bounds = np.zeros((n, 2), dtype=int)
    bounds[:,1] = 1 * a

    res = linprog(c=c, A_ub=A, b_ub=b, bounds=bounds)

    if not res.success:
        match (res.status):
            case 1:
                print("iteration limit reached")
            case 2:
                print("infeasible: please try with feasible discharges")
            case 3:
                print("problem unbounded")
            case 4:
                print("numerical difficulties")
        return
    
    optimal_control = res.x
    optimal_cost = res.fun
    
    return (res.x, res.fun)

In [159]:
# smoke test
S, C = 1, 0.8
x0 = 0
delta = np.ones(5)
d = np.zeros(5)
d[3:] = 1.0
carbon_cost = np.array([5.0, 3.0, 1.0, 2.0, 1.0])
a = np.ones(5)

print(optimal_discharge(x0, S, C, delta, d, carbon_cost, a))

(array([0. , 0. , 0.8, 0.2, 1. ]), 2.2)


In [160]:
class PiecewiseConstant:
    def __init__(self, default, change_times, values):
        # change_times[i] is when value switches to values[i]
        self.events = list(zip(change_times, values))
        self.default = default
        self._idx = 0
        self.current = default

    def advance(self, t):
        """Update current value if t matches next event."""
        if self._idx < len(self.events) and self.events[self._idx][0] == t:
            self.current = self.events[self._idx][1]
            self._idx += 1

    def next_change(self, t_end):
        if self._idx < len(self.events):
            return self.events[self._idx][0]
        return t_end
    
def intervals_to_switch(t_intervals:np.ndarray, values: np.ndarray, default: float):
    assert t_intervals.shape[0] == values.size
    t_switch = t_intervals.flat
    values_switch = np.ones(2 * values.size) * default
    values_switch[::2] = values
    return (t_switch, values_switch)

def merge(
    carbon_t: np.ndarray,
    carbon_c: np.ndarray,
    d_t: np.ndarray | None = None, # array[tuple[]]
    d_p: np.ndarray | None = None,
    a: np.ndarray | None = None, # array[tuple[]]
    t_end: np.datetime64 | None = None,
):
    """Implementation of a sweep line."""
    if t_end is None:
        t_end = carbon_t[-1]
    if d_t is None or d_p is None:
        assert d_t == d_p
        d_t = np.array([])
        d_p = np.array([])
    if a is None:
        a = np.array([])

    d_s, d_v = intervals_to_switch(d_t, d_p, 0)
    a_s, a_v = intervals_to_switch(a, np.zeros(a.shape[0]), 1)
    d_pw = PiecewiseConstant(0, d_s, d_v)
    c_pw = PiecewiseConstant(0, carbon_t, carbon_c)
    a_pw = PiecewiseConstant(1.0, a_s, a_v)

    delta = []
    d = []
    carbon_cost = []
    available = []

    curr_t = carbon_t[0]
    signals = [d_pw, c_pw, a_pw]
    for s in signals:
        s.advance(curr_t)
    
    while curr_t < t_end:
        next_t = min(s.next_change(t_end) for s in signals)
        delta_t = (next_t - curr_t) / np.timedelta64(1, 'h')
    
        delta.append(delta_t)
        d.append(d_pw.current)
        carbon_cost.append(c_pw.current)
        available.append(a_pw.current)
        
        curr_t = next_t
        for s in signals:
            s.advance(curr_t)
    
    return {
        "delta": np.array(delta),
        "d": np.array(d),
        "carbon_cost": np.array(carbon_cost),
        "a": np.array(available),
    }

In [161]:
from datetime import datetime
# smoke test
carbon_t = np.array([datetime(2000, 1,1, 13,5),datetime(2000, 1,1, 14,5),datetime(2000, 1,1, 15,5)], dtype=np.datetime64)
carbon_c = np.array([1.0,2.0,3.0])
d_t = np.array([(datetime(2000, 1,1, 14,0),datetime(2000, 1,1, 15,0))], dtype=np.datetime64)
d_p = np.array([10.0])
a = np.array([(datetime(2000, 1,1, 14,30),datetime(2000, 1,1, 14,45))], dtype=np.datetime64)

merge(carbon_t, carbon_c, d_t, d_p, None, np.datetime64(datetime(2000,1,1,15,10)))

{'delta': array([0.91666667, 0.08333333, 0.91666667, 0.08333333, 0.08333333]),
 'd': array([ 0., 10., 10.,  0.,  0.]),
 'carbon_cost': array([1., 1., 2., 2., 3.]),
 'a': array([1., 1., 1., 1., 1.])}

In [ ]:
from datetime import time, date
from tqdm import tqdm
# Now we want to establish how to define a load
# discharge should be in d_t (intervals), d_p form
# availability should be in a_t (intervals), form

# And a carbon series
# carbon forecast should be in c_t (start), c_c form

def daily_to_datetime(t_intervals: list[tuple[time]], date: date | np.datetime64):
    if isinstance(date, np.datetime64):
         date = date.item()
    dt_intervals = []
    for s, t in t_intervals:
        start = datetime.combine(date, s)
        end = datetime.combine(date, t)
        dt_intervals.append((start, end))
    return np.array(dt_intervals, dtype=np.datetime64)

d_daily_t = [(time(8), time(9)), (time(17), time(18))]
d_daily_p = np.array([10,10])


for day, chunk in tqdm(df.groupby(df["from"].dt.floor("D"))):
    # remove fractional days
    if len(chunk)!= 48:
        continue
    c_t = chunk["from"].to_numpy()
    c_c = chunk["forecast"].to_numpy()
    d_t = daily_to_datetime(d_daily_t, day)
    res = merge(c_t,c_c, d_t, d_daily_p, t_end=chunk["to"].to_numpy()[-1])
    x, f = optimal_discharge(0, 100, 100, res["delta"], res["d"], res["carbon_cost"], res["a"])
    

  0%|          | 0/3081 [00:00<?, ?it/s]

100%|██████████| 3081/3081 [00:06<00:00, 463.89it/s]
